In [ ]:
%sql
USE CATALOG nyc_taxi;
USE SCHEMA bronze;
CREATE VOLUME IF NOT EXISTS  RAW_DATA;
DESCRIBE VOLUME RAW_DATA; 

In [ ]:
from pyspark.sql import functions as F

raw_path = "/Volumes/nyc_taxi/bronze/raw_data/*.parquet"
bronze_table = "nyc_taxi.bronze.green_taxi"

raw_files = spark.read.parquet(raw_path)
bronze_df = spark.read.table(bronze_table)

raw_count = raw_files.count()
bronze_count = bronze_df.count()

if raw_count == 0:
    raise AssertionError("The raw volume contains no parquet records")
if bronze_count == 0:
    raise AssertionError(f"Bronze table {bronze_table} contains no records")

# Auto Loader is incremental and the Bronze table may contain prior batches.
# Validate presence and schema rather than requiring global row-count equality.
missing_columns = sorted(set(raw_files.columns) - set(bronze_df.columns))
if missing_columns:
    raise AssertionError(f"Bronze table is missing source columns: {missing_columns}")

print(f"Ingestion validation passed: raw_records={raw_count:,}, bronze_records={bronze_count:,}")
